# Week 3: Contrastive Probe with Multi-Token Lookahead

## The Problem with Single-Token Classification

Previous attempts failed because we classified single tokens:
```
"The book I read last week was" + "The" → "...wasThe" → CODE? ❌
```

The probe sees `"wasThe"` and thinks it's a variable name!

## Multi-Token Lookahead Solution

Instead, we look ahead **2 tokens**:
```
"...was" + "The" → generate next → "Girl"
Classify: "...wasThe Girl" → LANGUAGE ✅

"...using" + "O" → generate next → "Auth"
Classify: "...usingO Auth" → CODE ✅
```

## Expected Results

- CODE Recall: ~85% (keep the good performance)
- False Positives: 7 → ~2 (fix the context issue)
- Overall Accuracy: 76.1% → **~87%+** ✅

---

In [1]:
# Cell 1: Install
!pip install -q transformers torch accelerate scipy scikit-learn pandas matplotlib seaborn

In [2]:
# Cell 2: Imports
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import entropy as scipy_entropy
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
print("✅ Imports complete")

✅ Imports complete


In [3]:
# Cell 3: Load Model
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    output_hidden_states=True
)
model.eval()

print(f"✅ Model loaded on {model.device}")

Loading codellama/CodeLlama-7b-Instruct-hf...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

✅ Model loaded on cuda:0


In [4]:
# Cell 4: Load training data

LABELED_DATA_FILE = '/content/training_data_combined_FIXED.csv'

print(f"Loading {LABELED_DATA_FILE}...")
df = pd.read_csv(LABELED_DATA_FILE)

# Validate and convert labels
valid_labels = df['label'].isin(['code', 'language'])
df = df[valid_labels]
df['label_binary'] = df['label'].map({'language': 0, 'code': 1})

code_count = (df['label_binary']==1).sum()
lang_count = (df['label_binary']==0).sum()

print(f"\n✅ Loaded {len(df)} training examples")
print(f"   LANGUAGE: {lang_count} ({lang_count/len(df)*100:.1f}%)")
print(f"   CODE: {code_count} ({code_count/len(df)*100:.1f}%)")
print(f"   Ratio: {lang_count}:{code_count} ({lang_count/code_count:.2f}:1)")

Loading /content/training_data_combined_FIXED.csv...

✅ Loaded 1780 training examples
   LANGUAGE: 1353 (76.0%)
   CODE: 427 (24.0%)
   Ratio: 1353:427 (3.17:1)


In [5]:
# Cell 5: Extract hidden states

SELECTED_LAYERS = [8, 16, 31]

def get_multi_layer_state(text: str, layers: List[int]) -> np.ndarray:
    """Extract and concatenate hidden states from multiple layers."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    states = [
        outputs.hidden_states[layer_idx + 1][:, -1, :].cpu().numpy()[0]
        for layer_idx in layers
    ]
    return np.concatenate(states).astype(np.float32)

print(f"Extracting hidden states from layers {SELECTED_LAYERS}...")

X_train = []
y_train = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting"):
    h = get_multi_layer_state(row['full_text'], SELECTED_LAYERS)
    X_train.append(h)
    y_train.append(row['label_binary'])

X_train = np.array(X_train)
y_train = np.array(y_train)

print(f"\n✅ Hidden states extracted: {X_train.shape}")

Extracting hidden states from layers [8, 16, 31]...


Extracting:   0%|          | 0/1780 [00:00<?, ?it/s]


✅ Hidden states extracted: (1780, 12288)


In [6]:
# Cell 6: Train probe

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

probe = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred_cv = cross_val_predict(probe, X_train_scaled, y_train, cv=cv)
cv_accuracy = accuracy_score(y_train, y_pred_cv)

print(f"\n{'='*80}")
print(f"PROBE TRAINING RESULTS")
print(f"{'='*80}")
print(f"\n5-Fold CV Accuracy: {cv_accuracy:.1%}")
print(f"\n{classification_report(y_train, y_pred_cv, target_names=['LANGUAGE', 'CODE'])}")

cm = confusion_matrix(y_train, y_pred_cv)
code_recall = cm[1,1] / (cm[1,0] + cm[1,1]) if (cm[1,0] + cm[1,1]) > 0 else 0
print(f"CODE Recall: {code_recall:.1%} ← Key metric")

probe.fit(X_train_scaled, y_train)
print(f"\n✅ Probe trained")


PROBE TRAINING RESULTS

5-Fold CV Accuracy: 95.4%

              precision    recall  f1-score   support

    LANGUAGE       0.98      0.96      0.97      1353
        CODE       0.88      0.94      0.91       427

    accuracy                           0.95      1780
   macro avg       0.93      0.95      0.94      1780
weighted avg       0.96      0.95      0.95      1780

CODE Recall: 93.9% ← Key metric

✅ Probe trained


## Multi-Token Lookahead Implementation

In [7]:
# Cell 7: Multi-token lookahead helper functions

def softmax(logits: np.ndarray) -> np.ndarray:
    logits_stable = logits - np.max(logits)
    exp_logits = np.exp(logits_stable)
    return exp_logits / np.sum(exp_logits)

def entropy_from_probs(probs: np.ndarray) -> float:
    if len(probs) == 0 or np.sum(probs) == 0:
        return 0.0
    norm_probs = probs / np.sum(probs)
    return float(scipy_entropy(norm_probs, base=2))

def get_next_token_prediction(text: str) -> str:
    """Get the most likely next token after text."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0, -1, :].cpu().numpy()

    next_token_id = np.argmax(logits)
    return tokenizer.decode([next_token_id])

def classify_token_type(text: str) -> Tuple[int, float]:
    """Classify: 1=code, 0=language."""
    h = get_multi_layer_state(text, SELECTED_LAYERS).reshape(1, -1)
    h_scaled = scaler.transform(h)
    token_type = probe.predict(h_scaled)[0]
    probability = probe.predict_proba(h_scaled)[0, 1]
    return int(token_type), float(probability)

# ✨ NEW: Multi-token lookahead
LOOKAHEAD_TOKENS = 2  # Look ahead 2 tokens
CODE_MASS_THRESHOLD = 0.30  # Slightly higher threshold with better context

def analyze_candidate_tokens_lookahead(
    prompt: str,
    top_k: int = 10,
    lookahead: int = 2,
    verbose: bool = False
) -> Dict:
    """
    Multi-token lookahead analysis.

    Instead of classifying: prompt + token
    We classify: prompt + token + next_token + next_next_token

    This gives much better context!
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0, -1, :].cpu().numpy()

    probs = softmax(logits)
    top_indices = np.argsort(probs)[-top_k:][::-1]

    candidates = []
    code_votes = 0
    lang_votes = 0

    for idx in top_indices:
        token = tokenizer.decode([idx])
        prob = probs[idx]

        # ✨ MULTI-TOKEN LOOKAHEAD
        # Build sequence: prompt + token + next_token + next_next_token
        sequence = prompt + token

        for _ in range(lookahead - 1):  # lookahead-1 because we already have 1 token
            next_token = get_next_token_prediction(sequence)
            sequence += next_token

        # Now classify the FULL SEQUENCE (much better context!)
        token_type, type_prob = classify_token_type(sequence)

        candidates.append({
            'token': token,
            'prob': prob,
            'lookahead_sequence': sequence,  # For debugging
            'type': 'CODE' if token_type == 1 else 'LANGUAGE',
            'type_prob': type_prob
        })

        if token_type == 1:
            code_votes += prob
        else:
            lang_votes += prob

    is_code_uncertainty = code_votes > CODE_MASS_THRESHOLD

    if verbose:
        print(f"\nMulti-token lookahead analysis (lookahead={lookahead}):")
        for c in candidates:
            # Show the lookahead sequence
            lookahead_part = c['lookahead_sequence'].replace(prompt, '')
            print(f"  '{c['token']}' (p={c['prob']:.3f}) → ...{lookahead_part}... → {c['type']}")
        print(f"\nVotes: CODE={code_votes:.3f}, LANGUAGE={lang_votes:.3f}")
        print(f"Threshold: {CODE_MASS_THRESHOLD}")
        print(f"Decision: {'CODE uncertainty - STOP' if is_code_uncertainty else 'LANGUAGE uncertainty - CONTINUE'}")

    return {
        'candidates': candidates,
        'code_votes': code_votes,
        'lang_votes': lang_votes,
        'is_code_uncertainty': is_code_uncertainty,
        'code_mass_ratio': code_votes / (code_votes + lang_votes) if (code_votes + lang_votes) > 0 else 0
    }

print(f"✅ Multi-token lookahead functions ready")
print(f"   LOOKAHEAD_TOKENS = {LOOKAHEAD_TOKENS}")
print(f"   CODE_MASS_THRESHOLD = {CODE_MASS_THRESHOLD}")

✅ Multi-token lookahead functions ready
   LOOKAHEAD_TOKENS = 2
   CODE_MASS_THRESHOLD = 0.3


In [8]:
# Cell 8: Contrastive generation with multi-token lookahead

def generate_with_lookahead(
    prompt: str,
    entropy_threshold: float = 3.0,
    top_k_candidates: int = 10,
    lookahead: int = 2,
    max_tokens: int = 50,
    verbose: bool = True
) -> Dict:
    """
    Contrastive generation with MULTI-TOKEN LOOKAHEAD.

    At high entropy:
    - Get top-K candidates
    - For each: simulate next N tokens
    - Classify the full sequence
    - Vote based on sequences (not single tokens)
    """
    current_text = prompt
    generated_token_ids = []
    entropy_trace = []
    stop_reason = None
    stop_info = {}

    if verbose:
        print(f"\n{'='*80}")
        print(f"Prompt: '{prompt}'")
        print(f"Multi-token lookahead: {lookahead} tokens")
        print(f"{'='*80}")

    for step in range(max_tokens):
        inputs = tokenizer(current_text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits[0, -1, :].cpu().numpy()

        probs = softmax(logits)
        H = entropy_from_probs(probs)
        entropy_trace.append(H)

        next_token_id = np.argmax(probs)
        next_token = tokenizer.decode([next_token_id])

        if verbose:
            print(f"\nStep {step + 1}: '{next_token}' H={H:.2f}")

        if H > entropy_threshold:
            if verbose:
                print(f"  ⚠️  HIGH ENTROPY - analyzing with {lookahead}-token lookahead...")

            analysis = analyze_candidate_tokens_lookahead(
                current_text,
                top_k=top_k_candidates,
                lookahead=lookahead,
                verbose=verbose
            )

            if analysis['is_code_uncertainty']:
                if verbose:
                    print(f"  ❗ CODE UNCERTAINTY - STOPPING!")
                stop_reason = "code_uncertainty"
                stop_info = {
                    'step': step,
                    'entropy': H,
                    'code_votes': analysis['code_votes'],
                    'lang_votes': analysis['lang_votes'],
                    'code_mass_ratio': analysis['code_mass_ratio']
                }
                break
            else:
                if verbose:
                    print(f"  ✓ LANGUAGE uncertainty - continuing")

        generated_token_ids.append(next_token_id)
        current_text += next_token

        if next_token_id == tokenizer.eos_token_id:
            stop_reason = "eos"
            break

    if stop_reason is None:
        stop_reason = "max_tokens"

    generated_text = tokenizer.decode(generated_token_ids, skip_special_tokens=True)

    if verbose:
        print(f"\n{'='*80}")
        print(f"Stop: {stop_reason}")
        print(f"Generated: '{generated_text}'")
        print(f"{'='*80}")

    return {
        'prompt': prompt,
        'generated_text': generated_text,
        'full_text': prompt + generated_text,
        'entropy_trace': entropy_trace,
        'stop_reason': stop_reason,
        'stop_info': stop_info,
        'num_steps': len(generated_token_ids)
    }

print("✅ Multi-token lookahead generation ready")

✅ Multi-token lookahead generation ready


## Testing

In [9]:
# Cell 9: Demo test

print("\n" + "="*80)
print("DEMO: Multi-Token Lookahead")
print("="*80)

test_prompts = [
    "For our API, JWT tokens are signed using",  # Should STOP
    "The book I read last week was",             # Should CONTINUE (was failing before)
]

for test_prompt in test_prompts:
    result = generate_with_lookahead(
        test_prompt,
        entropy_threshold=3.0,
        lookahead=2,
        max_tokens=10,
        verbose=True
    )
    print("\n" + "-"*80 + "\n")


DEMO: Multi-Token Lookahead

Prompt: 'For our API, JWT tokens are signed using'
Multi-token lookahead: 2 tokens

Step 1: 'a' H=3.66
  ⚠️  HIGH ENTROPY - analyzing with 2-token lookahead...

Multi-token lookahead analysis (lookahead=2):
  'a' (p=0.309) → ...aprivate... → LANGUAGE
  'the' (p=0.189) → ...theprivate... → LANGUAGE
  'R' (p=0.157) → ...RSA... → CODE
  'an' (p=0.071) → ...anR... → CODE
  'H' (p=0.061) → ...HMA... → CODE
  'our' (p=0.034) → ...ourprivate... → LANGUAGE
  '`' (p=0.018) → ...`RS... → CODE
  '[' (p=0.016) → ...[R... → CODE
  'private' (p=0.013) → ...privatekeys... → LANGUAGE
  'this' (p=0.010) → ...thisprivate... → LANGUAGE

Votes: CODE=0.322, LANGUAGE=0.555
Threshold: 0.3
Decision: CODE uncertainty - STOP
  ❗ CODE UNCERTAINTY - STOPPING!

Stop: code_uncertainty
Generated: ''

--------------------------------------------------------------------------------


Prompt: 'The book I read last week was'
Multi-token lookahead: 2 tokens

Step 1: 'The' H=7.34
  ⚠️  HIGH E

In [10]:
# Cell 10: Full test suite

CODE_TEST_CASES = [
    {'prompt': 'In our React app, authentication is done using', 'category': 'auth_method'},
    {'prompt': 'In the backend, passwords are hashed with', 'category': 'auth_hash'},
    {'prompt': 'For our API, JWT tokens are signed using', 'category': 'auth_signing'},
    {'prompt': 'In production, the OAuth provider we use is', 'category': 'auth_provider'},
    {'prompt': 'On the server, session data is stored in', 'category': 'session_store'},
    {'prompt': 'For data persistence, the database we use is', 'category': 'database_type'},
    {'prompt': 'In the application, we query the database using', 'category': 'database_query'},
    {'prompt': 'For database access, the ORM library is', 'category': 'database_orm'},
    {'prompt': 'To improve performance, caching is implemented with', 'category': 'database_cache'},
    {'prompt': 'For the REST API, the framework we use is', 'category': 'web_framework'},
    {'prompt': 'In production, the web server runs on', 'category': 'web_server'},
    {'prompt': 'In the client code, HTTP requests are made using', 'category': 'http_client'},
    {'prompt': 'For data fetching, our GraphQL server uses', 'category': 'graphql_server'},
    {'prompt': 'For the UI, the frontend framework is', 'category': 'frontend_framework'},
    {'prompt': 'In the application, state management is handled by', 'category': 'frontend_state'},
    {'prompt': 'For the interface, components are built with', 'category': 'frontend_components'},
    {'prompt': 'In the SPA, routing is done using', 'category': 'frontend_routing'},
    {'prompt': 'For training, the model is trained with', 'category': 'ml_framework'},
    {'prompt': 'In our neural network, deep learning is implemented using', 'category': 'ml_deep_learning'},
    {'prompt': 'For gradient descent, the optimizer we use is', 'category': 'ml_optimizer'},
    {'prompt': 'For hosting, we deploy to', 'category': 'cloud_platform'},
    {'prompt': 'In Kubernetes, containers are orchestrated with', 'category': 'cloud_containers'},
    {'prompt': 'For automation, the CI/CD pipeline uses', 'category': 'cloud_cicd'},
    {'prompt': 'In the test suite, unit tests are written with', 'category': 'test_unit'},
    {'prompt': 'For building assets, the bundler we use is', 'category': 'build_bundler'},
    {'prompt': 'For dependencies, package management is done with', 'category': 'build_package_manager'},
]

LANGUAGE_TEST_CASES = [
    {'prompt': 'The weather today is', 'category': 'description_weather'},
    {'prompt': 'The meeting yesterday was', 'category': 'description_meeting'},
    {'prompt': 'My favorite color has always been', 'category': 'description_color'},
    {'prompt': 'The book I read last week was', 'category': 'description_book'},
    {'prompt': 'The movie we watched seemed', 'category': 'description_movie'},
    {'prompt': 'The main idea of the story is to', 'category': 'explanation_idea'},
    {'prompt': 'The cooking process works by', 'category': 'explanation_process'},
    {'prompt': 'This teaching approach helps to', 'category': 'explanation_approach'},
    {'prompt': 'The benefit of exercise is', 'category': 'explanation_benefit'},
    {'prompt': 'When installing furniture in my home, you should', 'category': 'instruction_furniture'},
    {'prompt': 'To debug a relationship problem, first', 'category': 'instruction_debug'},
    {'prompt': 'Before deploying troops, the general needs to', 'category': 'instruction_deploy'},
    {'prompt': 'The configuration of the room requires', 'category': 'instruction_config'},
    {'prompt': 'To optimize your morning routine, try', 'category': 'instruction_optimize'},
    {'prompt': 'The framework of the argument is', 'category': 'nontechnical_framework'},
    {'prompt': 'My mental state is managed by', 'category': 'nontechnical_state'},
    {'prompt': 'The library in town has', 'category': 'nontechnical_library'},
    {'prompt': 'Running the business takes', 'category': 'nontechnical_running'},
    {'prompt': 'The function of the heart is', 'category': 'nontechnical_function'},
    {'prompt': 'Implementing the new policy will', 'category': 'nontechnical_implement'},
]

ALL_TEST_CASES = [
    {**case, 'expected_stopped': True} for case in CODE_TEST_CASES
] + [
    {**case, 'expected_stopped': False} for case in LANGUAGE_TEST_CASES
]

print(f"\n{'='*80}")
print(f"TESTING MULTI-TOKEN LOOKAHEAD")
print(f"{'='*80}")
print(f"\nTotal: {len(ALL_TEST_CASES)} test cases")
print(f"Lookahead: {LOOKAHEAD_TOKENS} tokens")
print(f"Threshold: {CODE_MASS_THRESHOLD}")

print(f"\n🔄 Running tests...\n")

test_results = []

for test_case in tqdm(ALL_TEST_CASES, desc="Testing"):
    result = generate_with_lookahead(
        test_case['prompt'],
        entropy_threshold=3.0,
        lookahead=LOOKAHEAD_TOKENS,
        max_tokens=20,
        verbose=False
    )

    stopped_for_code = result['stop_reason'] == 'code_uncertainty'
    expected_stopped = test_case['expected_stopped']
    correct = stopped_for_code == expected_stopped

    max_entropy = np.max(result['entropy_trace']) if result['entropy_trace'] else 0

    test_results.append({
        'prompt': test_case['prompt'],
        'category': test_case['category'],
        'expected_stopped': expected_stopped,
        'stopped_for_code': stopped_for_code,
        'correct': correct,
        'stop_reason': result['stop_reason'],
        'max_entropy': max_entropy,
        'generated_text': result.get('generated_text', ''),
        'num_tokens_generated': result.get('num_steps', 0)
    })

df_results = pd.DataFrame(test_results)

print(f"\n{'='*80}")
print(f"MULTI-TOKEN LOOKAHEAD RESULTS")
print(f"{'='*80}")

overall_accuracy = df_results['correct'].mean()
print(f"\n📊 OVERALL ACCURACY: {overall_accuracy:.1%}")

print(f"\n📈 BY CLASS:")
for expected_val in [True, False]:
    class_name = "CODE (should stop)" if expected_val else "LANGUAGE (should not stop)"
    subset = df_results[df_results['expected_stopped'] == expected_val]
    accuracy = subset['correct'].mean() if len(subset) > 0 else 0
    correct = subset['correct'].sum()
    total = len(subset)
    stopped_count = subset['stopped_for_code'].sum()
    print(f"\n   {class_name}")
    print(f"      Correct: {correct}/{total}")
    print(f"      Accuracy: {accuracy:.1%}")
    print(f"      Stopped for code: {stopped_count}/{total}")

tp = len(df_results[(df_results['expected_stopped']==True) & (df_results['stopped_for_code']==True)])
fp = len(df_results[(df_results['expected_stopped']==False) & (df_results['stopped_for_code']==True)])
fn = len(df_results[(df_results['expected_stopped']==True) & (df_results['stopped_for_code']==False)])
tn = len(df_results[(df_results['expected_stopped']==False) & (df_results['stopped_for_code']==False)])

print(f"\n🎯 CONFUSION MATRIX")
print(f"                       Predicted NO STOP    Predicted STOPPED")
print(f"Expected NO STOP             {tn:<12}          {fp:<12}")
print(f"Expected STOP                {fn:<12}          {tp:<12}")

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"\n📐 METRICS")
print(f"   Precision: {precision:.1%}")
print(f"   Recall: {recall:.1%}")
print(f"   F1-Score: {f1:.1%}")

errors = df_results[~df_results['correct']]
if len(errors) > 0:
    print(f"\n❌ ERRORS ({len(errors)}/{len(df_results)}):")
    for idx, error in errors.iterrows():
        exp = "SHOULD STOP" if error['expected_stopped'] else "SHOULD NOT STOP"
        act = "STOPPED" if error['stopped_for_code'] else "DID NOT STOP"
        print(f"\n   '{error['prompt']}'")
        print(f"      Expected: {exp}, Actual: {act}")
else:
    print(f"\n🎉 PERFECT! No errors!")

print(f"\n{'='*80}")
print(f"COMPARISON")
print(f"{'='*80}")
print(f"\n| Metric           | Original | Single-Token | Multi-Token |")
print(f"|------------------|----------|--------------|-------------|")
print(f"| Overall Accuracy | 82.6%    | 76.1% ❌     | {overall_accuracy:.1%} {'✅' if overall_accuracy > 0.826 else '⚠️'} |")
print(f"| CODE Recall      | 80.8%    | 84.6%        | {recall:.1%} {'✅' if recall > 0.808 else '⚠️'} |")
print(f"| LANG Precision   | 85.0%    | 65.0% ❌     | {tn/(tn+fp)*100 if (tn+fp) > 0 else 0:.1f}% {'✅' if tn/(tn+fp) > 0.85 else '⚠️'} |")
print(f"| Total Errors     | 8        | 11 ❌        | {len(errors)} {'✅' if len(errors) < 8 else '⚠️'} |")
print(f"\n{'='*80}")


TESTING MULTI-TOKEN LOOKAHEAD

Total: 46 test cases
Lookahead: 2 tokens
Threshold: 0.3

🔄 Running tests...



Testing:   0%|          | 0/46 [00:00<?, ?it/s]


MULTI-TOKEN LOOKAHEAD RESULTS

📊 OVERALL ACCURACY: 87.0%

📈 BY CLASS:

   CODE (should stop)
      Correct: 22/26
      Accuracy: 84.6%
      Stopped for code: 22/26

   LANGUAGE (should not stop)
      Correct: 18/20
      Accuracy: 90.0%
      Stopped for code: 2/20

🎯 CONFUSION MATRIX
                       Predicted NO STOP    Predicted STOPPED
Expected NO STOP             18                    2           
Expected STOP                4                     22          

📐 METRICS
   Precision: 91.7%
   Recall: 84.6%
   F1-Score: 88.0%

❌ ERRORS (6/46):

   'In production, the web server runs on'
      Expected: SHOULD STOP, Actual: DID NOT STOP

   'For training, the model is trained with'
      Expected: SHOULD STOP, Actual: DID NOT STOP

   'For hosting, we deploy to'
      Expected: SHOULD STOP, Actual: DID NOT STOP

   'In Kubernetes, containers are orchestrated with'
      Expected: SHOULD STOP, Actual: DID NOT STOP

   'To optimize your morning routine, try'
      Expected: